# Egg Sex Classification — Data Preprocessing

This notebook processes raw LSCI egg images into train/validation/test pickle files for use in model training.

**Pipeline:**
1. Load CSV ground-truth labels
2. Split images by egg ID (75% train / 12.5% val / 12.5% test)
3. Convert 16-bit images → 8-bit, center-crop, zoom to brightest spot, resize
4. Save processed data as pickle files

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!unzip '/content/drive/MyDrive/CS 163/Export_dataset_November24.zip' -d ./unzipped

In [ ]:
import os
import csv
import math
import random
import pickle
from pathlib import Path
from random import shuffle

import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torchvision.transforms as transforms

random.seed(1839)

## 2. Exploratory Data Analysis

Visualize the images with the lowest pixel intensities to identify potential low-quality images before processing.

In [ ]:
def sort_images_by_intensity(directories):
    """Sort all PNG images across the given directories by mean pixel intensity."""
    image_data = []
    for directory in directories:
        for filename in os.listdir(directory):
            if filename.endswith('.png'):
                filepath = os.path.join(directory, filename)
                img = cv2.imread(filepath, cv2.IMREAD_GRAYSCALE)
                if img is not None:
                    image_data.append((filename, filepath, np.mean(img)))
    image_data.sort(key=lambda x: x[2])
    return image_data


def visualize_images(image_data, num_images=30, cols=6):
    """Display a grid of images with their file names and mean intensities."""
    rows = math.ceil(num_images / cols)
    plt.figure(figsize=(cols * 3, rows * 3))
    for i in range(min(num_images, len(image_data))):
        filename, filepath, mean_intensity = image_data[i]
        img = cv2.imread(filepath, cv2.IMREAD_GRAYSCALE)
        plt.subplot(rows, cols, i + 1)
        plt.imshow(img, cmap='gray')
        plt.title(f"{filename}\nIntensity: {mean_intensity:.1f}", fontsize=7)
        plt.axis('off')
    plt.tight_layout()
    plt.show()


sorted_images = sort_images_by_intensity([
    './unzipped/HH19/Lower_quality',
    './unzipped/HH25/Lower_quality',
])
visualize_images(sorted_images, num_images=30)

## 3. Load Ground-Truth Labels

Build a list of data-point dicts from the CSV files. Each entry contains the image path, binary sex label (0=male, 1=female), and metadata (egg ID, grade, stage).

In [ ]:
def load_data_points(dir_to_data, csv_path):
    """
    Parse a ground-truth CSV and return a list of data-point dicts.

    CSV columns (0-indexed):
        0: image number, 1: label, 2: egg ID, 3: grade, 4: real, 5: stage, 6: sub-folder ('.' = root)
    """
    data_points = []
    with open(csv_path, 'r') as f:
        reader = csv.reader(f)
        next(reader)  # skip header
        for row in reader:
            sub_folder = row[6]
            img_name = f'img_{row[0]}.png' if sub_folder == '.' else f'lowQ_img_{row[0]}.png'
            data_points.append({
                'label': int(row[1]),
                'path': os.path.join(dir_to_data, sub_folder, img_name),
                'additional_info': {
                    'id':    int(row[2]),
                    'grade': int(row[3]),
                    'real':  int(row[4]),
                    'stage': int(row[5]),
                },
            })
    # Keep only images that exist on disk
    data_points = [dp for dp in data_points if os.path.exists(dp['path'])]
    print(f'Loaded {len(data_points)} data points from {csv_path}')
    return data_points


hh19_data = load_data_points('./unzipped/HH19/', './GT_labels_eggs_HH19.csv')
hh25_data = load_data_points('./unzipped/HH25/', './GT_labels_eggs_HH25.csv')

## 4. Image Processing Functions

In [ ]:
def zoom_to_brightest(image, crop_radius=1200):
    """
    Crop a square region of radius `crop_radius` around the brightest pixel
    within the central 60% of the image, then resize back to the original size.
    This centres the laser speckle pattern in the field of view.
    """
    img = np.array(image)
    h, w = img.shape

    # Search only the central 60% to avoid edge artefacts
    x0, x1 = int(h * 0.20), int(h * 0.80)
    y0, y1 = int(w * 0.20), int(w * 0.80)
    roi = img[x0:x1, y0:y1]

    # Find brightest pixel in the ROI and map back to full-image coordinates
    rx, ry = np.unravel_index(np.argmax(roi), roi.shape)
    cx, cy = rx + x0, ry + y0

    # Crop around that centre
    cropped = img[
        max(0, cx - crop_radius): min(h, cx + crop_radius),
        max(0, cy - crop_radius): min(w, cy + crop_radius),
    ]
    return Image.fromarray(cropped).resize(image.size, Image.LANCZOS)


def load_and_process_image(file_path, img_size=640):
    """
    Full preprocessing pipeline for a single image:
      1. Read 16-bit PNG and convert to 8-bit
      2. Center-crop to a square
      3. Zoom to the brightest spot
      4. Resize to (img_size × img_size)
    Returns a PIL Image, or None on error.
    """
    try:
        image = cv2.imread(file_path, cv2.IMREAD_UNCHANGED)
        image = (image / 256).astype(np.uint8)
        image = Image.fromarray(image)

        shorter_edge = min(image.size)
        pipeline = transforms.Compose([
            transforms.CenterCrop(shorter_edge),
            transforms.Lambda(zoom_to_brightest),
            transforms.Resize((img_size, img_size)),
        ])
        return pipeline(image)
    except Exception as e:
        print(f'Error processing {file_path}: {e}')
        return None

## 5. Train / Validation / Test Split

Images are split **by egg ID** so that all images of the same egg end up in the same split (≈75% train / 12.5% val / 12.5% test). This prevents data leakage between splits.

In [ ]:
def split_by_id(data_points, seed=1839):
    """
    Split data points into train / validation / test sets.
    All images sharing the same egg ID are kept in the same partition.
    Target ratio: 75% train, 12.5% val, 12.5% test.
    """
    random.seed(seed)

    # Build {egg_id: count} and shuffle
    id_counts = {}
    for dp in data_points:
        eid = dp['additional_info']['id']
        id_counts[eid] = id_counts.get(eid, 0) + 1
    id_counts = list(id_counts.items())
    id_counts.sort()
    shuffle(id_counts)

    total = sum(c for _, c in id_counts)
    target_val = math.ceil(0.125 * total)
    target_test = math.ceil(0.125 * total)

    # Greedily fill test, then val, then train
    def fill_partition(id_counts, target):
        selected, remaining, count = [], [], 0
        for item in id_counts:
            if count < target:
                selected.append(item); count += item[1]
            else:
                remaining.append(item)
        return selected, remaining

    test_ids_list,  rem       = fill_partition(id_counts, target_test)
    val_ids_list,   train_ids_list = fill_partition(rem, target_val)

    test_ids  = {eid for eid, _ in test_ids_list}
    val_ids   = {eid for eid, _ in val_ids_list}
    train_ids = {eid for eid, _ in train_ids_list}

    train = [dp for dp in data_points if dp['additional_info']['id'] in train_ids]
    val   = [dp for dp in data_points if dp['additional_info']['id'] in val_ids]
    test  = [dp for dp in data_points if dp['additional_info']['id'] in test_ids]

    n = len(train) + len(val) + len(test)
    print(f'  train={len(train)/n:.1%}  val={len(val)/n:.1%}  test={len(test)/n:.1%}')
    return train, val, test, train_ids, val_ids, test_ids

In [ ]:
random.seed(1839)

# ── HH19 split (stratified by sex) ──────────────────────────────────────────
hh19_male   = [dp for dp in hh19_data if dp['label'] == 0]
hh19_female = [dp for dp in hh19_data if dp['label'] == 1]

print('HH19 male:')
train_m19, val_m19, test_m19, train_ids_m19, val_ids_m19, test_ids_m19 = split_by_id(hh19_male)
print('HH19 female:')
train_f19, val_f19, test_f19, train_ids_f19, val_ids_f19, test_ids_f19 = split_by_id(hh19_female)

hh19_train = train_m19 + train_f19
hh19_val   = val_m19   + val_f19
hh19_test  = test_m19  + test_f19

# ── HH25 split: IDs shared with HH19 follow the same partition ──────────────
def split_by_id_with_hh19(data_points, train_ids_hh19, val_ids_hh19, test_ids_hh19, seed=1839):
    """
    Like split_by_id, but egg IDs that appeared in HH19 are assigned to the
    matching HH19 partition to prevent cross-dataset leakage.
    """
    random.seed(seed)

    id_counts = {}
    for dp in data_points:
        eid = dp['additional_info']['id']
        id_counts[eid] = id_counts.get(eid, 0) + 1
    id_counts = list(id_counts.items())
    id_counts.sort(); shuffle(id_counts)

    # Separate IDs that already have a HH19 assignment
    carry_train = [(eid, c) for eid, c in id_counts if eid in train_ids_hh19]
    carry_val   = [(eid, c) for eid, c in id_counts if eid in val_ids_hh19]
    carry_test  = [(eid, c) for eid, c in id_counts if eid in test_ids_hh19]
    remaining   = [(eid, c) for eid, c in id_counts
                   if eid not in train_ids_hh19 | val_ids_hh19 | test_ids_hh19]

    total = sum(c for _, c in remaining)
    target_val  = math.ceil(0.125 * total)
    target_test = math.ceil(0.125 * total)

    def fill_partition(items, target):
        selected, rest, count = [], [], 0
        for item in items:
            if count < target:
                selected.append(item); count += item[1]
            else:
                rest.append(item)
        return selected, rest

    new_test,  rem     = fill_partition(remaining, target_test)
    new_val,   new_train = fill_partition(rem, target_val)

    train_ids = {eid for eid, _ in carry_train + new_train}
    val_ids   = {eid for eid, _ in carry_val   + new_val}
    test_ids  = {eid for eid, _ in carry_test  + new_test}

    train = [dp for dp in data_points if dp['additional_info']['id'] in train_ids]
    val   = [dp for dp in data_points if dp['additional_info']['id'] in val_ids]
    test  = [dp for dp in data_points if dp['additional_info']['id'] in test_ids]

    n = len(train) + len(val) + len(test)
    print(f'  train={len(train)/n:.1%}  val={len(val)/n:.1%}  test={len(test)/n:.1%}')
    return train, val, test

hh25_male   = [dp for dp in hh25_data if dp['label'] == 0]
hh25_female = [dp for dp in hh25_data if dp['label'] == 1]

print('HH25 male:')
train_m25, val_m25, test_m25 = split_by_id_with_hh19(hh25_male,   train_ids_m19, val_ids_m19, test_ids_m19)
print('HH25 female:')
train_f25, val_f25, test_f25 = split_by_id_with_hh19(hh25_female, train_ids_f19, val_ids_f19, test_ids_f19)

hh25_train = train_m25 + train_f25
hh25_val   = val_m25   + val_f25
hh25_test  = test_m25  + test_f25

## 6. Process Images and Save to Pickle

Images are processed in chunks to keep memory usage manageable. Each chunk is saved as a separate pickle file containing a list of `{label, image, additional_info}` dicts.

In [ ]:
IMG_SIZE         = 640
PICKLE_CHUNK_SIZE = 5000


def save_split_to_pickle(data_points, split_name, dataset_name, img_size=IMG_SIZE, chunk_size=PICKLE_CHUNK_SIZE):
    """Process images and save them to pickle files under ./processed_data/<dataset>/<split>/."""
    output_dir = Path(f'./processed_data/{dataset_name}/{split_name}')
    output_dir.mkdir(parents=True, exist_ok=True)

    for chunk_no in range(0, len(data_points), chunk_size):
        chunk = data_points[chunk_no: chunk_no + chunk_size]
        processed = []
        for i, dp in enumerate(chunk):
            if (i + 1) % 100 == 0:
                print(f'  [{split_name}] chunk {chunk_no // chunk_size + 1}: {i + 1}/{len(chunk)}')
            image = load_and_process_image(dp['path'], img_size)
            processed.append({'label': dp['label'], 'image': image, 'additional_info': dp['additional_info']})

        part = chunk_no // chunk_size + 1
        out_path = output_dir / f'{split_name}_{img_size}_part_{part}.pickle'
        with open(out_path, 'wb') as f:
            pickle.dump(processed, f)
        print(f'  Saved {out_path}')


print('Processing HH19...')
save_split_to_pickle(hh19_train, 'training',   'HH19')
save_split_to_pickle(hh19_val,   'validation', 'HH19')
save_split_to_pickle(hh19_test,  'testing',    'HH19')

print('\nProcessing HH25...')
save_split_to_pickle(hh25_train, 'training',   'HH25')
save_split_to_pickle(hh25_val,   'validation', 'HH25')
save_split_to_pickle(hh25_test,  'testing',    'HH25')

print('\nDone.')

## 7. Save Processed Data to Drive

In [ ]:
!zip -r /content/processed_data.zip /content/processed_data
from google.colab import files
# Optionally copy to Drive instead:
# !cp /content/processed_data.zip '/content/drive/MyDrive/CS 163/processed_data.zip'